# 4. The real words, and the demo

1. **The real test, once** (set `RUN_REAL_TEST = True` in the first cell): the seven trained
   runs read the 100 handwritten words (`WORK/real/real_test.tar`), greedily and with
   the language model at the weights chosen on the synthetic validation set. Writes
   `results/phase3_real_<run>.json`, and per run `real_predictions.tsv` and
   `real_errors.png` (the misread words) in its folder.
2. **The scores** of the seven runs, side by side.
3. **The demo**: of the final recipe's two seeds, the one better on synthetic validation,
   exported to ONNX (checked against PyTorch on validation words) with the language model
   for the browser: the Hugging Face model repository and the static Space, built in
   `WORK/hf`. Writes `results/phase3_demo.json`.
4. **Upload** (set `UPLOAD = True`): the model repository and the Space to Hugging Face.
   Needs a Hugging Face token with write access as a Colab secret called `HF_TOKEN` (key
   icon on the left; allow this notebook to read it). Never paste the token into a cell.

No GPU needed (100 words). Inputs on Drive: `WORK/runs/<run>/best.pt`,
`WORK/results/phase2_val_<run>.json` (and `phase2_test_<run>.json`), `WORK/lm/char_lm.pkl`,
`WORK/synth/val.tar`, and the real words `WORK/real/real_test.tar` (cut with
`scripts/cut_writing_pages.py`). Afterwards `collect_results.ipynb` gathers the results in one zip.

Run the cells in order. Section 1 reads the real words once: a run already read is not
read again.

In [ ]:
# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-recognition"
REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
BRANCHES = ["main"]   # the branch to clone
REAL_SET = f"{WORK}/real/real_test.tar"   # the 100 handwritten words, as cut
# every trained run: the second round (final recipe: the first two), then the first round
RUNS = ["round2_convnext_tummhcd", "round2_convnext_tummhcd_seed1", "round2_convnext_imagenet",
        "round2_crnn_scratch", "convnext_tummhcd", "convnext_imagenet", "crnn_scratch"]
FINAL = ["round2_convnext_tummhcd", "round2_convnext_tummhcd_seed1"]   # the demo is the better on validation
HF_ID = "Chingkheinganba/handwritten-meitei-mayek-word-recognition"      # model repository and Space
RUN_REAL_TEST = False   # True once: every run reads the real words, one time
UPLOAD = False          # True to put the model and the demo on Hugging Face

In [ ]:
import glob, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")

def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")

REPO_DIR = "/content/repo"
url = f"https://github.com/{REPO}.git"
if not os.path.exists(f"{REPO_DIR}/scripts/build_demo.py"):
    for branch in BRANCHES:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f"{REPO_DIR}/scripts/build_demo.py"):
            break
    else:
        raise SystemExit("no branch in BRANCHES has the code. git: " + r.stderr)
else:  # a runtime left from an earlier session: bring the code up to date
    branch = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    run("git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", url, branch)
    run("git", "-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
run(sys.executable, "-m", "pip", "install", "-q", "timm", "huggingface_hub", "scipy", "onnx", "onnxruntime", "pymupdf")

missing = [f for f in ("lm/char_lm.pkl", "synth/val.tar") if not os.path.exists(f"{WORK}/{f}")]
missing += [f"runs/{r}/best.pt" for r in RUNS if not os.path.exists(f"{WORK}/runs/{r}/best.pt")]
missing += [f"results/phase2_val_{r}.json" for r in RUNS if not os.path.exists(f"{WORK}/results/phase2_val_{r}.json")]
if not os.path.exists(REAL_SET):
    missing.append(f"{REAL_SET} (the set cut by scripts/cut_writing_pages.py)")
if missing:
    raise SystemExit("missing on Drive:\n  " + "\n  ".join(missing))
for f in ("lm/char_lm.pkl", "synth/val.tar"):
    os.makedirs(os.path.dirname(f"/content/{f}"), exist_ok=True)
    shutil.copy(f"{WORK}/{f}", f"/content/{f}")      # local copies read faster than Drive
os.makedirs("/content/real", exist_ok=True)
shutil.copy(REAL_SET, "/content/real/real_test.tar")
os.makedirs(f"{WORK}/results", exist_ok=True)
LM, REAL = "/content/lm/char_lm.pkl", "/content/real/real_test.tar"

from mayek_htr.data import FixedSet
real = FixedSet(REAL)
print(f"code: {subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()}")
print(f"real words: {len(real)} ({', '.join(f'{k} {real.kinds.count(k)}' for k in sorted(set(real.kinds)))})")

## 1. The real test, once

Every run reads the 100 real words with the settings it was given on the synthetic
validation set (`--tuned`): greedy decoding, and beam search with the language model. The
test is used once: a run whose `phase3_real_<run>.json` is on Drive is not read again.

In [ ]:
if not RUN_REAL_TEST:
    print("RUN_REAL_TEST is False: nothing is read. Set it to True in the first cell, run the first two cells "
          "again, then this one.")
else:
    for r in RUNS:
        out = f"{WORK}/results/phase3_real_{r}.json"
        if os.path.exists(out):
            print(f"{r}: already read ({out}), not read again")
            continue
        stream(sys.executable, "scripts/eval_recogniser.py", "--checkpoint", f"{WORK}/runs/{r}/best.pt",
               "--set", REAL, "--lm", LM, "--tuned", f"{WORK}/results/phase2_val_{r}.json", "--out", out,
               "--predictions", f"{WORK}/runs/{r}/real_predictions.tsv", "--sheet", f"{WORK}/runs/{r}/real_errors.png")

## 2. The scores

Character and word error rates on the 100 real words, without and with the language model;
the words read right with their 95% interval; the confusable pairs (how often one
character was read as the other, both ways). Below the table: the misread words of the
final recipe's runs.

In [ ]:
import pandas as pd
from IPython.display import Image as Show, display

rows = []
for r in RUNS:
    f = f"{WORK}/results/phase3_real_{r}.json"
    if not os.path.exists(f):
        continue
    d = json.load(open(f))
    g, w = d["greedy"], d["with_lm"]
    lo, hi = w["word_accuracy_95ci"]
    rows.append({"run": r, "words": w["words"], "CER": round(100 * g["cer"], 2), "WER": round(100 * g["wer"], 1),
                 "CER with LM": round(100 * w["cer"], 2), "WER with LM": round(100 * w["wer"], 1),
                 "words right (95% CI)": f"{100 * w['word_accuracy']:.0f}% ({100 * lo:.0f}-{100 * hi:.0f})",
                 "pairs with LM (right / written, swapped)": ", ".join(
                     f"{k} {v['correct']}/{v['reference']}, {v['read_as_the_other']}" for k, v in w.get("pairs", {}).items())})
if rows:
    print("error rates in %")
    display(pd.DataFrame(rows))
    both = [x for x in rows if x["run"] in FINAL]
    if len(both) == 2:
        print("final recipe, mean of the two seeds: CER with LM {:.2f}%, WER with LM {:.1f}%".format(
            sum(x["CER with LM"] for x in both) / 2, sum(x["WER with LM"] for x in both) / 2))
    for r in FINAL:
        if os.path.exists(f"{WORK}/runs/{r}/real_errors.png"):
            print(r)
            display(Show(f"{WORK}/runs/{r}/real_errors.png"))
else:
    print("no real-word results yet: run section 1 with RUN_REAL_TEST = True")

## 3. The demo

The final recipe's run that is better on synthetic validation (lower CER with the language
model, then WER), exported for the browser. The model repository (`WORK/hf/model`: the
network, the language model, the decoding weights, the scores, the model card) and the
static Space (`WORK/hf/space`: the page, which downloads the network and the language model
from the model repository). The cards quote the synthetic test and, once section 1 has run,
the real words.

In [ ]:
stream(sys.executable, "scripts/build_demo.py", "--runs", f"{WORK}/runs", "--results", f"{WORK}/results",
       "--candidates", *FINAL, "--lm", LM, "--check-set", "/content/synth/val.tar",
       "--model-dir", f"{WORK}/hf/model", "--space-dir", f"{WORK}/hf/space", "--hf-id", HF_ID,
       "--summary", f"{WORK}/results/phase3_demo.json")
for d in ("model", "space"):
    print(f"\n{d}:")
    for p in sorted(Path(f"{WORK}/hf/{d}").rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(f'{WORK}/hf/{d}')}  {p.stat().st_size / 1e6:.1f} MB")

## 4. Upload to Hugging Face

The model repository first (the page downloads the network and the language model from
it), then the Space. Both are public. Afterwards the demo is at
`https://huggingface.co/spaces/<HF_ID>`; the first visit downloads about 40 MB, later ones
come from the browser's cache.

In [ ]:
if not UPLOAD:
    print("UPLOAD is False: nothing is uploaded. Set it to True in the first cell, run the first two cells, "
          "section 3, then this one.")
else:
    from google.colab import userdata
    from huggingface_hub import HfApi
    try:
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        raise SystemExit("Add a Hugging Face token with write access as a Colab secret called HF_TOKEN (key icon "
                         "on the left) and allow this notebook to use it, then run this cell again.")
    api = HfApi(token=hf_token)
    api.create_repo(HF_ID, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=f"{WORK}/hf/model", repo_id=HF_ID, repo_type="model",
                      commit_message="Word recogniser, language model and browser files")
    api.create_repo(HF_ID, repo_type="space", space_sdk="static", exist_ok=True)
    api.upload_folder(folder_path=f"{WORK}/hf/space", repo_id=HF_ID, repo_type="space",
                      commit_message="Browser demo")
    print(f"model: https://huggingface.co/{HF_ID}")
    print(f"demo:  https://huggingface.co/spaces/{HF_ID}")

## 5. Update the cards only

Rewrites the model card, the Space card, the paragraph under the demo page and
`results.json` from the results in the repository, and uploads them (set `UPDATE_CARDS = True`
below). The network and the language model are not touched, so this needs no GPU and no
files on Drive besides the repository: use it when a result changes after the demo was built.
The token comes from the Colab secret `HF_TOKEN`.

In [ ]:
UPDATE_CARDS = False   # True to rewrite and upload the cards
if UPDATE_CARDS:
    from google.colab import userdata
    env = dict(os.environ, HF_TOKEN=userdata.get("HF_TOKEN"))
    r = subprocess.run([sys.executable, "scripts/update_cards.py", "--results", "results", "--out",
                        "/content/hf_cards", "--hf-id", HF_ID, "--upload"], capture_output=True, text=True, env=env)
    print(r.stdout + r.stderr)
else:
    print("UPDATE_CARDS is False: nothing is uploaded.")

## Afterwards

`collect_results.ipynb` gathers `results/phase3_*.json` and, per run, `real_predictions.tsv`
and `real_errors.png` in one zip.